# v5 — drop Hinglish bridge, queryify YT, keep Aksharantar

Same v3 recipe (per ADR 014): **MNR loss + binary positives + batch 32 + 2 epochs + best-on-dev checkpointing + fp16 AMP + gradient checkpointing**. Only the pair set differs.

**v5 composition (11,827 pairs)** — incorporates all v4 ablation learnings:
- drop Hinglish bridge (v4c regressed −2.7%)
- drop raw YT chunks (v4a regressed −6.8% — monologue ≠ query)
- keep filtered Aksharantar (3,660 — the carry source in v4b)
- add 339 query-ified YT pairs (DeepSeek rewrites of mechanic monologue as user queries)

**Just Run all.** No cell edits needed. Pair dataset + corpus files already set up (upload `graph.db` + `benchmark_dev.json` once via Files panel if not already there).

**Decision gate:** v5 graded nDCG@10 ≥ 0.561 (v3 0.535 + 5%) → promote as v4. Else we either stop or iterate further.

---

*For future ablation runs, change `VARIANT` in the cell below to `'v4a'`, `'v4b'`, `'v4c'`, or a new variant name.*

In [ ]:
# ==== current run ====
VARIANT = 'v5'   # 'v4a' | 'v4b' | 'v4c' | 'v5'
# =====================

In [ ]:
!pip install -q sentence-transformers==3.4.1 datasets==3.2.0 huggingface_hub==0.27.0

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset
from collections import Counter

PAIRS_REPO = f'ManmohanBuildsProducts/auto-parts-search-pairs-{VARIANT}'
print(f'variant: {VARIANT}  repo: {PAIRS_REPO}')
ds = load_dataset(PAIRS_REPO, split='train')
positives = ds.filter(lambda r: r['label'] >= 1.0)
print(f'loaded {len(ds)} records, {len(positives)} positives')
print(Counter(positives['pair_type']).most_common())

## Upload corpus files manually via Colab Files panel

Upload these two from your Mac (`data/knowledge_graph/graph.db`, `data/training/golden/benchmark_dev.json`). Cell below will halt otherwise.

In [ ]:
import json, os, sqlite3
from collections import defaultdict
if not os.path.exists('graph.db') or not os.path.exists('benchmark_dev.json'):
    raise SystemExit('UPLOAD graph.db and benchmark_dev.json via the Files panel, then re-run this cell.')

conn = sqlite3.connect('graph.db')
parts = list(conn.execute("SELECT id, name FROM nodes WHERE type='part'"))
aliases = defaultdict(list)
for alias_name, part_id in conn.execute(
    "SELECT n.name, e.dst FROM edges e JOIN nodes n ON n.id = e.src "
    "WHERE e.type='known_as' AND n.type='alias'"
): aliases[part_id].append(alias_name)
systems = defaultdict(list)
for part_id, sys_name in conn.execute(
    "SELECT e.src, n.name FROM edges e JOIN nodes n ON n.id = e.dst "
    "WHERE e.type='in_system' AND n.type='system'"
): systems[part_id].append(sys_name)
conn.close()

part_ids, doc_texts, rel_strings = [], [], []
for pid, name in parts:
    al = aliases.get(pid, []); sy = systems.get(pid, [])
    doc = name
    if al: doc += ' | ' + ', '.join(al)
    if sy: doc += ' | system: ' + ', '.join(sy)
    part_ids.append(pid); doc_texts.append(doc)
    rel_strings.append({name.lower(), *(a.lower() for a in al)})

print(f'corpus: {len(part_ids)} parts')
benchmark_dev = json.load(open('benchmark_dev.json'))
print(f'dev: {len(benchmark_dev)} queries')

In [ ]:
import random
rng = random.Random(42)
by_type = defaultdict(list)
for q in benchmark_dev: by_type[q['query_type']].append(q)
dev_subset = []
for qt, qs in sorted(by_type.items()):
    s = list(qs); rng.shuffle(s); dev_subset.extend(s[:7])
print(f'dev subset: {len(dev_subset)}')

from sentence_transformers.evaluation import InformationRetrievalEvaluator
queries_map = {f'q{i}': q['query'] for i, q in enumerate(dev_subset)}
corpus_map = {pid: t for pid, t in zip(part_ids, doc_texts)}
relevant_docs = {}
for i, q in enumerate(dev_subset):
    expected = [ep.lower().strip() for ep in q.get('expected_parts', []) if ep.strip()]
    rels = set()
    for pid, rel_str_set in zip(part_ids, rel_strings):
        for ep in expected:
            if any(ep in rs or rs in ep for rs in rel_str_set):
                rels.add(pid); break
    relevant_docs[f'q{i}'] = rels

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries_map, corpus=corpus_map, relevant_docs=relevant_docs,
    name=f'dev_subset_{VARIANT}',
    mrr_at_k=[10], ndcg_at_k=[10], accuracy_at_k=[1,5,10], precision_recall_at_k=[5,10],
    show_progress_bar=False, corpus_chunk_size=5000,
)
print('evaluator ready')

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

random.seed(42)
examples = [InputExample(texts=[r['text_a'], r['text_b']]) for r in positives]
random.shuffle(examples)
BATCH = 32
loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
print(f'{len(examples)} examples, {len(loader)} steps/epoch @ batch {BATCH}')

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses

print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
print(f'total VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

model = SentenceTransformer('BAAI/bge-m3')
model[0].auto_model.gradient_checkpointing_enable()   # saves ~30% VRAM; ~25% slower backward
model.max_seq_length = 128
train_loss = losses.MultipleNegativesRankingLoss(model)

EPOCHS = 2
WARMUP = int(len(loader) * EPOCHS * 0.1)
CHECKPOINT_STEPS = max(100, len(loader) // 3)
print(f'epochs: {EPOCHS}  warmup: {WARMUP}  checkpoint_steps: {CHECKPOINT_STEPS}')

In [ ]:
OUT_DIR = f'auto-parts-search-{VARIANT}'
CHKPT_DIR = f'{VARIANT}-checkpoints'

model.fit(
    train_objectives=[(loader, train_loss)],
    evaluator=ir_evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={'lr': 2e-5},
    output_path=OUT_DIR,
    save_best_model=True,
    evaluation_steps=CHECKPOINT_STEPS,
    checkpoint_path=CHKPT_DIR,
    checkpoint_save_steps=CHECKPOINT_STEPS,
    checkpoint_save_total_limit=6,
    show_progress_bar=True,
    use_amp=True,
)
print(f'trained. peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

In [ ]:
from huggingface_hub import HfApi
MODEL_REPO = f'ManmohanBuildsProducts/auto-parts-search-{VARIANT}'
api = HfApi()
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUT_DIR, repo_id=MODEL_REPO,
    commit_message=f'{VARIANT} — BGE-m3 + MNR, 2ep best-on-dev, {len(examples)} pairs',
)
print(f'pushed: https://huggingface.co/{MODEL_REPO}')

## Next (manual)

Change `VARIANT` above to the next letter, **Runtime → Disconnect and delete runtime**, re-attach T4, Run all. Repeat until A/B/C all trained and on HF.

Then locally:
```bash
# joint-pool judge over v3 + v4a + v4b + v4c
python3.11 -m scripts.judge_benchmark \
    --benchmark data/training/golden/benchmark_dev.json \
    --model ManmohanBuildsProducts/auto-parts-search-v3 \
    --model ManmohanBuildsProducts/auto-parts-search-v4a \
    --model ManmohanBuildsProducts/auto-parts-search-v4b \
    --model ManmohanBuildsProducts/auto-parts-search-v4c \
    --deepseek-model deepseek-chat \
    --out data/training/experiments/2026-04-14-v4/joint_pool.jsonl

# rescore each model
for v in v3 v4a v4b v4c; do
  python3.11 -m training.evaluate_graded \
      --model ManmohanBuildsProducts/auto-parts-search-${v} \
      --graded data/training/experiments/2026-04-14-v4/joint_pool.jsonl \
      --out data/training/experiments/2026-04-14-v4/${v}.json
done
```
